In [3]:
import time
import pandas as pd
from transformers import pipeline

DEBERTA_MODEL_NAME = (
    "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
)

deberta_classifier = pipeline(
    task="zero-shot-classification",
    model=DEBERTA_MODEL_NAME
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [9]:
deberta_predictions = []
deberta_confidences = []
deberta_latencies = []

In [6]:
evaluation_df = pd.read_csv(
    "data/job_posts_evaluation.csv"
)

evaluation_df

,job_description,expected_category
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines
2,Build machine learning models for text classif...,artificial intelligence and machine learning
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation
5,Create automated release pipelines using Docke...,devops and deployment automation
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering
8,Monitor security events using SIEM tools and i...,cybersecurity and information security
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security


In [11]:
deberta_predictions = []
deberta_confidences = []
deberta_latencies = []

In [13]:
candidate_labels = [
    "data engineering and data pipelines",
    "artificial intelligence and machine learning",
    "devops and deployment automation",
    "cloud infrastructure and cloud engineering",
    "cybersecurity and information security",
    "software and application development",
]

In [14]:
for job_description in evaluation_df["job_description"]:
    start_time = time.perf_counter()

    result = deberta_classifier(
        job_description,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    latency = time.perf_counter() - start_time

    deberta_predictions.append(result["labels"][0])
    deberta_confidences.append(result["scores"][0] * 100)
    deberta_latencies.append(latency)

In [16]:
deberta_results_df = evaluation_df.copy()

deberta_results_df["predicted_category"] = deberta_predictions
deberta_results_df["confidence_percent"] = deberta_confidences
deberta_results_df["latency_seconds"] = deberta_latencies

deberta_results_df["is_correct"] = (
    deberta_results_df["expected_category"]
    == deberta_results_df["predicted_category"]
)

In [17]:
deberta_results_df[
    [
        "job_description",
        "expected_category",
        "predicted_category",
        "confidence_percent",
        "latency_seconds",
        "is_correct",
    ]
]

,job_description,expected_category,predicted_category,confidence_percent,latency_seconds,is_correct
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines,data engineering and data pipelines,75.736141,3.392815,True
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines,data engineering and data pipelines,98.355561,3.214391,True
2,Build machine learning models for text classif...,artificial intelligence and machine learning,artificial intelligence and machine learning,59.960335,2.987544,True
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning,software and application development,43.592760,2.987059,False
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation,devops and deployment automation,72.821289,3.413115,True
5,Create automated release pipelines using Docke...,devops and deployment automation,devops and deployment automation,72.304702,3.110395,True
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering,cloud infrastructure and cloud engineering,88.351709,3.655696,True
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering,cloud infrastructure and cloud engineering,54.613847,3.705221,True
8,Monitor security events using SIEM tools and i...,cybersecurity and information security,cybersecurity and information security,96.525186,3.296955,True
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security,cybersecurity and information security,89.051938,3.445063,True


In [18]:
deberta_accuracy = (
    deberta_results_df["is_correct"].mean() * 100
)

deberta_average_latency = (
    deberta_results_df["latency_seconds"].mean()
)

deberta_average_confidence = (
    deberta_results_df["confidence_percent"].mean()
)

print(f"Accuracy: {deberta_accuracy:.2f}%")
print(f"Average latency: {deberta_average_latency:.3f} seconds")
print(f"Average confidence: {deberta_average_confidence:.2f}%")

Accuracy: 91.67%
Average latency: 3.406 seconds
Average confidence: 78.80%
